In [0]:
%run ../0-common/env-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_table"

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
    .withColumn("session_type", F.lit("RACE"))
    .drop("race_date", "race_name", "ingestion_timestamp", "source_file")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("race_date", "race_name", "ingestion_timestamp", "source_file")
)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_result_df =  (
    results_sprints_df
    .withColumn("is_win", F.col("final_position") == 1)
    .withColumn("is_podium", F.col("final_position").between(1,3))
    .withColumn("has_points", F.col("points") > 0)
)

In [0]:
(
    fact_session_result_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)